# 3D Loss Landscape and SGD Trajectory

Visualize a two-dimensional slice of the MNIST MLP's loss surface and overlay an SGD
trajectory. To keep the demonstration tractable and the slice well-defined:

- Load the trained model from `models/mnist_mlp.npz`.
- Freeze its first layer and fine-tune only the output layer on a small MNIST training subset.
- Derive two parameter-space directions from the output-layer SGD trajectory using SVD/PCA.
- Evaluate the loss on a separate, fixed probe subset from MNIST's training split.

This is a slice through the output-layer parameter space, not the full network loss landscape.
The official test split is not used.

In [ ]:
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from tensorflow.keras.datasets import mnist

SEED = 42
rng = np.random.default_rng(SEED)

PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find pyproject.toml. Start Jupyter inside the project directory."
    )

ARTIFACT_PATH = PROJECT_ROOT / "models" / "mnist_mlp.npz"
REPORT_DIR = PROJECT_ROOT / "reports" / "figures"

print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifact: {ARTIFACT_PATH}")

Project root: c:\Users\hp\Documents\GitHub\multilayer-perceptron-from-scratch
Model artifact: c:\Users\hp\Documents\GitHub\multilayer-perceptron-from-scratch\models\mnist_mlp_256_128.npz


## Load model parameters and training data

Run `01_MLP_From_Scratch.ipynb` first to create the model artifact.
Only MNIST's official training split is loaded in this notebook.

In [16]:
if not ARTIFACT_PATH.is_file():
    raise FileNotFoundError(
        f"Model artifact not found: {ARTIFACT_PATH}\n"
        "Run notebooks/01_MLP_From_Scratch.ipynb to train and save the model."
    )

with np.load(ARTIFACT_PATH, allow_pickle=False) as artifact:
    required_keys = {
        "W1",
        "b1",
        "W2",
        "b2",
        "input_dim",
        "hidden_dim",
        "output_dim",
        "pixel_scale",
    }
    missing_keys = required_keys.difference(artifact.files)
    if missing_keys:
        raise ValueError(f"Model artifact is missing keys: {sorted(missing_keys)}")

    params = {
        name: artifact[name].astype(np.float32, copy=True)
        for name in ("W1", "b1", "W2", "b2")
    }
    input_dim = int(artifact["input_dim"])
    hidden_dim = int(artifact["hidden_dim"])
    output_dim = int(artifact["output_dim"])
    pixel_scale = float(artifact["pixel_scale"])

assert input_dim == 784
assert output_dim == 10
assert params["W1"].shape == (input_dim, hidden_dim)
assert params["b1"].shape == (hidden_dim,)
assert params["W2"].shape == (hidden_dim, output_dim)
assert params["b2"].shape == (output_dim,)
assert pixel_scale > 0
assert all(np.isfinite(value).all() for value in params.values())

(X_train, y_train), _ = mnist.load_data()

assert X_train.shape == (60_000, 28, 28)
assert y_train.shape == (60_000,)

print(f"Loaded model: {input_dim} → {hidden_dim} → {output_dim}")
print(f"Training images: {X_train.shape}")

ValueError: Model artifact is missing keys: ['hidden_dim']

## Prepare optimization and probe subsets

Fine-tune on one small subset, and evaluate the loss surface on a separate fixed probe subset.
Both subsets come from the official training split. This keeps the surface evaluation consistent
across parameter settings without using the official test set.

In [ ]:
TRAIN_SIZE = 1_024
PROBE_SIZE = 256

selected_indices = rng.permutation(len(y_train))[: TRAIN_SIZE + PROBE_SIZE]
train_indices = selected_indices[:TRAIN_SIZE]
probe_indices = selected_indices[TRAIN_SIZE:]

X_train_subset = (
    X_train[train_indices].reshape(TRAIN_SIZE, -1).astype(np.float32)
    / np.float32(pixel_scale)
)
y_train_subset = y_train[train_indices].astype(np.int64)

X_probe = (
    X_train[probe_indices].reshape(PROBE_SIZE, -1).astype(np.float32)
    / np.float32(pixel_scale)
)
y_probe = y_train[probe_indices].astype(np.int64)


def hidden_features(X: np.ndarray) -> np.ndarray:
    """Apply the frozen first layer and ReLU activation."""
    Z1 = X @ params["W1"] + params["b1"]
    return np.maximum(Z1, 0)


H_train = hidden_features(X_train_subset)
H_probe = hidden_features(X_probe)

print(f"Training feature matrix: {H_train.shape}")
print(f"Probe feature matrix:    {H_probe.shape}")

## Define the output-layer objective

The output parameters are represented as one vector containing flattened `W2` followed by `b2`.
The hidden features remain fixed throughout optimization and surface evaluation.

In [ ]:
def pack_output_params(W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    return np.concatenate([W2.ravel(), b2.ravel()])


def unpack_output_params(
    vector: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    weight_count = hidden_dim * output_dim
    W2 = vector[:weight_count].reshape(hidden_dim, output_dim)
    b2 = vector[weight_count:].reshape(output_dim)
    return W2, b2


def cross_entropy_from_logits(
    logits: np.ndarray,
    labels: np.ndarray,
) -> float:
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    log_normalizer = np.log(np.sum(np.exp(shifted), axis=1))
    row_indices = np.arange(len(labels))
    losses = log_normalizer - shifted[row_indices, labels]
    return float(np.mean(losses))


def output_layer_loss(
    vector: np.ndarray,
    features: np.ndarray,
    labels: np.ndarray,
) -> float:
    W2, b2 = unpack_output_params(vector)
    logits = features @ W2 + b2
    return cross_entropy_from_logits(logits, labels)


def output_layer_gradients(
    features: np.ndarray,
    labels: np.ndarray,
    W2: np.ndarray,
    b2: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    batch_size = len(labels)
    logits = features @ W2 + b2

    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_shifted = np.exp(shifted)
    probabilities = exp_shifted / exp_shifted.sum(axis=1, keepdims=True)

    probabilities[np.arange(batch_size), labels] -= 1.0
    probabilities /= batch_size

    dW2 = features.T @ probabilities
    db2 = probabilities.sum(axis=0)
    return dW2, db2


initial_vector = pack_output_params(params["W2"], params["b2"])
initial_probe_loss = output_layer_loss(initial_vector, H_probe, y_probe)
print(f"Initial probe loss: {initial_probe_loss:.4f}")

## Fine-tune the output layer and record SGD states

Each point in the trajectory records the output-layer weights and biases after an update.
The first-layer parameters remain frozen, so every recorded state belongs to the same
parameter subspace used by the loss-surface visualization.

In [ ]:
EPOCHS = 4
BATCH_SIZE = 128
LEARNING_RATE = 0.05

W2 = params["W2"].copy()
b2 = params["b2"].copy()

trajectory_vectors = [pack_output_params(W2, b2)]
trajectory_probe_losses = [
    output_layer_loss(trajectory_vectors[0], H_probe, y_probe)
]
epoch_train_losses = []
epoch_probe_losses = []

step_rng = np.random.default_rng(SEED)

for epoch in range(1, EPOCHS + 1):
    epoch_indices = step_rng.permutation(TRAIN_SIZE)

    for start in range(0, TRAIN_SIZE, BATCH_SIZE):
        batch_indices = epoch_indices[start : start + BATCH_SIZE]
        H_batch = H_train[batch_indices]
        y_batch = y_train_subset[batch_indices]

        dW2, db2 = output_layer_gradients(H_batch, y_batch, W2, b2)
        W2 -= LEARNING_RATE * dW2
        b2 -= LEARNING_RATE * db2

        current_vector = pack_output_params(W2, b2)
        trajectory_vectors.append(current_vector)
        trajectory_probe_losses.append(
            output_layer_loss(current_vector, H_probe, y_probe)
        )

    current_vector = pack_output_params(W2, b2)
    epoch_train_loss = output_layer_loss(
        current_vector,
        H_train,
        y_train_subset,
    )
    epoch_probe_loss = output_layer_loss(current_vector, H_probe, y_probe)

    epoch_train_losses.append(epoch_train_loss)
    epoch_probe_losses.append(epoch_probe_loss)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train-subset loss {epoch_train_loss:.4f} | "
        f"probe loss {epoch_probe_loss:.4f}"
    )

trajectory_vectors = np.asarray(trajectory_vectors, dtype=np.float32)
trajectory_probe_losses = np.asarray(trajectory_probe_losses, dtype=np.float32)

print(f"Recorded parameter states: {len(trajectory_vectors)}")
print(f"Parameters per state:      {trajectory_vectors.shape[1]:,}")

## Derive two directions from the trajectory

Center the recorded parameter states and use SVD to find the two directions that explain
the most variation in this output-layer trajectory. Project the trajectory into that plane.

In [ ]:
trajectory_mean = trajectory_vectors.mean(axis=0)
centered_trajectory = trajectory_vectors - trajectory_mean

_, singular_values, right_vectors = np.linalg.svd(
    centered_trajectory,
    full_matrices=False,
)

if right_vectors.shape[0] < 2:
    raise RuntimeError("The trajectory does not contain two independent directions.")

directions = right_vectors[:2].astype(np.float32)
trajectory_coordinates = (trajectory_vectors - initial_vector) @ directions.T

variance = singular_values**2
explained_variance = variance[:2] / variance.sum() if variance.sum() else np.zeros(2)

print(
    "Variance explained by plotted directions: "
    f"{explained_variance.sum():.1%}"
)
print(
    "Direction 1 / 2: "
    f"{explained_variance[0]:.1%} / {explained_variance[1]:.1%}"
)

## Evaluate the loss surface

Evaluate cross-entropy over a grid around the observed trajectory coordinates.
The surface uses the fixed probe subset and the frozen hidden features.

In [ ]:
GRID_SIZE = 31
RANGE_PADDING = 0.20

axis_ranges = []
for axis in range(2):
    values = trajectory_coordinates[:, axis]
    lower = float(values.min())
    upper = float(values.max())
    span = upper - lower

    if span < 1e-8:
        center = float(values.mean())
        span = max(abs(center), 1e-3)
        lower = center - span
        upper = center + span

    padding = span * RANGE_PADDING
    axis_ranges.append((lower - padding, upper + padding))

x_values = np.linspace(*axis_ranges[0], GRID_SIZE)
y_values = np.linspace(*axis_ranges[1], GRID_SIZE)
X_grid, Y_grid = np.meshgrid(x_values, y_values)
Z_grid = np.empty_like(X_grid, dtype=np.float32)

for row in range(GRID_SIZE):
    for column in range(GRID_SIZE):
        candidate = (
            initial_vector
            + X_grid[row, column] * directions[0]
            + Y_grid[row, column] * directions[1]
        )
        Z_grid[row, column] = output_layer_loss(candidate, H_probe, y_probe)

print(f"Surface grid: {GRID_SIZE} × {GRID_SIZE}")
print(f"Probe loss range: {Z_grid.min():.4f}–{Z_grid.max():.4f}")

## Plot the surface and SGD trajectory

The line shows each recorded SGD state projected into the two plotted directions.
Because the plotted space is a projection, any components outside the plane are not shown.

In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Surface(
        x=X_grid,
        y=Y_grid,
        z=Z_grid,
        colorscale="Viridis",
        opacity=0.82,
        colorbar=dict(title="Probe loss"),
        name="Loss surface",
    )
)

fig.add_trace(
    go.Scatter3d(
        x=trajectory_coordinates[:, 0],
        y=trajectory_coordinates[:, 1],
        z=trajectory_probe_losses,
        mode="lines+markers",
        marker=dict(
            size=3,
            color=np.arange(len(trajectory_probe_losses)),
            colorscale="Turbo",
            colorbar=dict(title="SGD step", x=1.08),
        ),
        line=dict(color="white", width=4),
        name="Projected SGD trajectory",
    )
)

fig.update_layout(
    title="MNIST output-layer loss slice and SGD trajectory",
    scene=dict(
        xaxis_title="PCA direction 1",
        yaxis_title="PCA direction 2",
        zaxis_title="Probe cross-entropy",
    ),
    width=1000,
    height=750,
)
fig.show()

## Inspect the trajectory in 2D

This plot shows how probe loss changed over the recorded SGD updates.
It complements the 3D view and makes the optimization trend easier to read.

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=np.arange(len(trajectory_probe_losses)),
        y=trajectory_probe_losses,
        mode="lines",
        name="Probe loss",
    )
)
fig.update_layout(
    title="Probe loss along the output-layer SGD trajectory",
    xaxis_title="Recorded update",
    yaxis_title="Cross-entropy loss",
)
fig.show()

## Save reusable visualization data

Save the surface grid, trajectory coordinates, and losses for reuse in the Streamlit app.
The generated files go under `reports/figures/`, which is intended for generated outputs.

In [ ]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

html_path = REPORT_DIR / "loss_surface.html"
data_path = REPORT_DIR / "loss_surface.npz"

fig.write_html(html_path, include_plotlyjs="cdn")

np.savez_compressed(
    data_path,
    x_grid=X_grid.astype(np.float32),
    y_grid=Y_grid.astype(np.float32),
    loss_grid=Z_grid,
    trajectory_coordinates=trajectory_coordinates.astype(np.float32),
    trajectory_probe_losses=trajectory_probe_losses,
    directions=directions,
    initial_output_params=initial_vector,
    explained_variance=explained_variance.astype(np.float32),
    train_size=np.array(TRAIN_SIZE),
    probe_size=np.array(PROBE_SIZE),
    epochs=np.array(EPOCHS),
    batch_size=np.array(BATCH_SIZE),
    learning_rate=np.array(LEARNING_RATE),
)

print(f"Interactive chart: {html_path}")
print(f"Reusable data:     {data_path}")

## Interpretation notes

- The plotted surface varies only output-layer weights and biases; the hidden layer is fixed.
- The two PCA directions are derived from this notebook's fine-tuning trajectory.
- The SGD path is projected into the plane, so movement perpendicular to that plane is omitted.
- The surface is evaluated on a small fixed training-derived probe subset; it illustrates local
  optimization behavior rather than serving as a generalization metric.
- The landscape can change with the model checkpoint, data subset, learning rate, and seed.

This visualization is intended to build intuition about optimization, not to characterize the
entire neural-network loss surface.